# Sentiment Autoresearch: Fine-tuning Qwen3.5 with Unsloth

This notebook applies Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern to **sentiment classification fine-tuning**. It:

1. Loads your labeled sentiment dataset (CSV/JSON/JSONL)
2. Fine-tunes **Qwen3.5-0.6B-Instruct** (or 1.7B) using **Unsloth + LoRA**
3. Evaluates classification accuracy / F1 on a held-out validation set
4. Runs an **autoresearch loop**: samples LoRA/training configs, keeps improvements, discards regressions
5. Exports the best model to **GGUF** for CPU deployment via Ollama / llama.cpp

**Why this works for packaged goods:** Sentiment in CPG is context-dependent — "bitter" is positive for coffee but negative for chocolate. A fine-tuned small model learns these domain-specific patterns from your labeled data.

**Requirements:** Colab with GPU (A100 recommended, T4 works for 0.6B), a labeled dataset with text + label columns.

---

## 1. Check GPU & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > GPU.")

In [ ]:
%%capture
# Install Unsloth (official Colab install method)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Qwen3.5 needs DeltaNet support
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
# Extra deps for eval & data
!uv pip install scikit-learn pandas openpyxl

## 2. Clone the Repo & Upload Your Data

The supporting code (data loading, evaluation, config) lives in the `sentiment/` package.

In [ ]:
import os

REPO_DIR = "/content/autoresearch-colab"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aigorahub/autoresearch-colab.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la sentiment/

### Upload Your Dataset

Upload a CSV, JSON, or JSONL file with at least two columns:
- **text**: The open-ended text response
- **label**: The sentiment label (e.g., "positive", "negative", "neutral")
- **category** (optional): The product category (e.g., "coffee", "chocolate")

In [ ]:
# Option A: Upload from your computer
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f"\nUploaded: {DATA_PATH}")

In [ ]:
# Option B: Google Drive (uncomment and edit)
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_PATH = "/content/drive/MyDrive/your_data.csv"

# Option C: Direct path (uncomment and edit)
# DATA_PATH = "/content/your_data.csv" 

## 3. Configure Your Dataset

In [ ]:
# ── EDIT THESE to match your data columns ──────────────────
TEXT_COL = "text"           # Column with the text responses
LABEL_COL = "label"         # Column with sentiment labels
CATEGORY_COL = None         # Set to column name if you have product categories
VAL_FRACTION = 0.15         # Fraction of data for validation
# ───────────────────────────────────────────────────────────

from sentiment.data import load_dataset, train_val_split, format_for_training

df = load_dataset(DATA_PATH, text_col=TEXT_COL, label_col=LABEL_COL, category_col=CATEGORY_COL)
LABELS = sorted(df["label"].unique().tolist())
print(f"\nDetected labels: {LABELS}")

train_df, val_df = train_val_split(df, val_fraction=VAL_FRACTION)

include_cat = CATEGORY_COL is not None and "category" in df.columns
train_data = format_for_training(train_df, LABELS, include_category=include_cat)
val_data = format_for_training(val_df, LABELS, include_category=include_cat)

print(f"\nTraining samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nExample conversation:")
for msg in train_data[0]["conversations"]:
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

## 4. Run Baseline Experiment

Trains the default config once to establish a baseline F1 score.

In [ ]:
import torch, gc, time, json, os
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from sentiment.config import DEFAULT_CONFIG, ExperimentConfig
from sentiment.evaluate import evaluate_model, print_eval_summary

def run_experiment(config, train_data, val_data, labels, experiment_dir="./experiments/current"):
    os.makedirs(experiment_dir, exist_ok=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        config.base_model,
        max_seq_length=config.max_seq_length,
        load_in_4bit=False,
        dtype=None,
        use_gradient_checkpointing="unsloth",
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=config.seed,
        use_rslora=False,
        loftq_config=None,
    )

    def formatting_func(examples):
        texts = []
        for convs in examples["conversations"]:
            text = tokenizer.apply_chat_template(
                convs, tokenize=False, add_generation_prompt=False,
            )
            texts.append(text)
        return {"text": texts}

    train_dataset = Dataset.from_list(train_data)
    train_dataset = train_dataset.map(
        formatting_func, batched=True,
        remove_columns=train_dataset.column_names,
    )

    FastLanguageModel.for_training(model)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        args=SFTConfig(
            per_device_train_batch_size=config.per_device_batch,
            gradient_accumulation_steps=config.grad_accum_steps,
            warmup_ratio=config.warmup_ratio,
            num_train_epochs=config.num_epochs,
            learning_rate=config.learning_rate,
            logging_steps=config.logging_steps,
            optim=config.optim,
            weight_decay=config.weight_decay,
            lr_scheduler_type=config.lr_scheduler,
            seed=config.seed,
            output_dir=experiment_dir,
            report_to="none",
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            max_seq_length=config.max_seq_length,
            dataset_text_field="text",
            packing=True,
        ),
    )

    t0 = time.time()
    trainer_stats = trainer.train()
    train_time = time.time() - t0
    peak_mem = torch.cuda.max_memory_reserved() / 1024**3

    print(f"Training time: {train_time:.1f}s | Peak VRAM: {peak_mem:.1f} GB")

    FastLanguageModel.for_inference(model)
    results = evaluate_model(model, tokenizer, val_data, labels)
    results["train_time"] = train_time
    results["peak_vram_gb"] = peak_mem
    results["config"] = config.to_dict()
    results["train_loss"] = trainer_stats.training_loss

    print_eval_summary(results)

    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()
    return results

print("=" * 60)
print("BASELINE EXPERIMENT")
print("=" * 60)
print(f"Config: {DEFAULT_CONFIG.describe()}")

baseline_results = run_experiment(DEFAULT_CONFIG, train_data, val_data, LABELS)
baseline_f1 = baseline_results["macro_f1"]
print(f"\n>>> BASELINE MACRO F1: {baseline_f1:.4f}")

## 5. Autoresearch Loop

Samples random LoRA / training configs, trains, evaluates, keeps improvements. Each iteration ~1-5 min.

**Adjust `MAX_EXPERIMENTS`** to control iterations. Set to `None` for infinite.

In [ ]:
import csv, traceback
from sentiment.config import sample_config

MAX_EXPERIMENTS = 30
N_MUTATIONS = 2
METRIC = "macro_f1"
RESULTS_FILE = "sentiment_results.tsv"

def log_result(results_file, exp_num, config, metric_val, status, extra=None):
    extra = extra or {}
    header_needed = not os.path.exists(results_file)
    with open(results_file, "a") as f:
        if header_needed:
            f.write("exp\tmetric\tstatus\ttrain_loss\ttrain_time\tvram_gb\t")
            f.write("model\tlora_r\tlora_alpha\tlr\tepochs\tbatch\t")
            f.write("grad_accum\tmax_seq\tscheduler\tnote\n")
        f.write(f"{exp_num}\t{metric_val:.4f}\t{status}\t")
        f.write(f"{extra.get('train_loss', 0):.4f}\t")
        f.write(f"{extra.get('train_time', 0):.1f}\t")
        f.write(f"{extra.get('peak_vram_gb', 0):.1f}\t")
        f.write(f"{config.base_model.split('/')[-1]}\t")
        f.write(f"{config.lora_r}\t{config.lora_alpha}\t")
        f.write(f"{config.learning_rate}\t{config.num_epochs}\t")
        f.write(f"{config.per_device_batch}\t{config.grad_accum_steps}\t")
        f.write(f"{config.max_seq_length}\t{config.lr_scheduler}\t")
        f.write(f"{extra.get('note', '')}\n")

best_config = DEFAULT_CONFIG
best_metric = baseline_f1
best_results = baseline_results

log_result(RESULTS_FILE, 0, DEFAULT_CONFIG, best_metric, "keep",
           {"train_loss": baseline_results.get("train_loss", 0),
            "train_time": baseline_results.get("train_time", 0),
            "peak_vram_gb": baseline_results.get("peak_vram_gb", 0),
            "note": "baseline"})

print(f"Baseline {METRIC}: {best_metric:.4f}")
print(f"Starting autoresearch loop ({MAX_EXPERIMENTS or 'infinite'} experiments)...")

n_kept = n_discarded = n_crashed = 0
experiment_num = 1

while MAX_EXPERIMENTS is None or experiment_num <= MAX_EXPERIMENTS:
    print(f"\n{'=' * 60}")
    print(f"[Exp {experiment_num}] best={best_metric:.4f} | kept={n_kept} | disc={n_discarded} | crash={n_crashed}")
    print("=" * 60)

    new_config = sample_config(baseline=best_config, n_changes=N_MUTATIONS)
    print(f"Config: {new_config.describe()}")

    try:
        results = run_experiment(
            new_config, train_data, val_data, LABELS,
            experiment_dir=f"./experiments/exp_{experiment_num:03d}",
        )
        metric_val = results[METRIC]

        if metric_val > best_metric:
            improvement = metric_val - best_metric
            print(f"\n>>> KEEP! {METRIC}={metric_val:.4f} (+{improvement:.4f})")
            best_metric = metric_val
            best_config = new_config
            best_results = results
            log_result(RESULTS_FILE, experiment_num, new_config, metric_val, "keep",
                       {"train_loss": results.get("train_loss", 0),
                        "train_time": results.get("train_time", 0),
                        "peak_vram_gb": results.get("peak_vram_gb", 0),
                        "note": "improvement"})
            n_kept += 1
        else:
            print(f"\n>>> DISCARD: {METRIC}={metric_val:.4f} (best={best_metric:.4f})")
            log_result(RESULTS_FILE, experiment_num, new_config, metric_val, "discard",
                       {"train_loss": results.get("train_loss", 0),
                        "train_time": results.get("train_time", 0),
                        "peak_vram_gb": results.get("peak_vram_gb", 0)})
            n_discarded += 1
    except Exception as e:
        print(f"\n>>> CRASH: {e}")
        traceback.print_exc()
        log_result(RESULTS_FILE, experiment_num, new_config, 0.0, "crash",
                   {"note": str(e)[:100]})
        n_crashed += 1
        gc.collect()
        torch.cuda.empty_cache()

    experiment_num += 1

print(f"\n{'=' * 60}")
print(f"DONE! Ran {experiment_num - 1} experiments.")
print(f"Best {METRIC}: {best_metric:.4f}")
print(f"Kept: {n_kept} | Discarded: {n_discarded} | Crashed: {n_crashed}")
print(f"Best config: {best_config.describe()}")

## 6. Analyze Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_results = pd.read_csv(RESULTS_FILE, sep="\t")
df_results["metric"] = pd.to_numeric(df_results["metric"], errors="coerce")

print(f"Total experiments: {len(df_results)}")
print(f"Outcomes: {df_results['status'].value_counts().to_dict()}")

kept = df_results[df_results["status"] == "keep"]
print("\nKept improvements:")
for _, row in kept.iterrows():
    print(f"  Exp {row['exp']:3.0f}: {METRIC}={row['metric']:.4f}  "
          f"r={row['lora_r']}  lr={row['lr']}  epochs={row['epochs']}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
valid = df_results[df_results["status"] != "crash"].reset_index(drop=True)
disc = valid[valid["status"] == "discard"]
ax.scatter(disc["exp"], disc["metric"], c="#cccccc", s=15, alpha=0.5, label="Discarded")
kept_v = valid[valid["status"] == "keep"]
ax.scatter(kept_v["exp"], kept_v["metric"], c="#2ecc71", s=60, zorder=4,
           label="Kept", edgecolors="black", linewidths=0.5)
if len(kept_v) > 0:
    running_max = kept_v["metric"].cummax()
    ax.step(kept_v["exp"], running_max, where="post", color="#27ae60",
            linewidth=2, alpha=0.7, label="Running best")
ax.set_xlabel("Experiment #")
ax.set_ylabel(f"{METRIC} (higher is better)")
ax.set_title("Autoresearch Progress")
ax.legend()
ax.grid(True, alpha=0.2)

ax = axes[1]
valid_lr = valid.dropna(subset=["lr", "metric"])
for status, group in valid_lr.groupby("status"):
    c = "#2ecc71" if status == "keep" else "#cccccc"
    ax.scatter(group["lr"], group["metric"], c=c, s=30, alpha=0.6, label=status.capitalize())
ax.set_xlabel("Learning Rate")
ax.set_ylabel(f"{METRIC}")
ax.set_xscale("log")
ax.set_title("Learning Rate vs Performance")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("sentiment_progress.png", dpi=150)
plt.show()

## 7. Train Final Model & Export to GGUF

Re-trains using the best config, then exports to GGUF for CPU deployment.

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print("=" * 60)
print(f"FINAL TRAINING with best config ({METRIC}={best_metric:.4f})")
print("=" * 60)

model, tokenizer = FastLanguageModel.from_pretrained(
    best_config.base_model,
    max_seq_length=best_config.max_seq_length,
    load_in_4bit=False,
    dtype=None,
    use_gradient_checkpointing="unsloth",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=best_config.lora_r,
    lora_alpha=best_config.lora_alpha,
    lora_dropout=best_config.lora_dropout,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=best_config.seed,
)

all_data = format_for_training(df, LABELS, include_category=include_cat)

def formatting_func(examples):
    texts = []
    for convs in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            convs, tokenize=False, add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

full_dataset = Dataset.from_list(all_data)
full_dataset = full_dataset.map(
    formatting_func, batched=True,
    remove_columns=full_dataset.column_names,
)

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=full_dataset,
    args=SFTConfig(
        per_device_train_batch_size=best_config.per_device_batch,
        gradient_accumulation_steps=best_config.grad_accum_steps,
        warmup_ratio=best_config.warmup_ratio,
        num_train_epochs=best_config.num_epochs,
        learning_rate=best_config.learning_rate,
        logging_steps=best_config.logging_steps,
        optim=best_config.optim,
        weight_decay=best_config.weight_decay,
        lr_scheduler_type=best_config.lr_scheduler,
        seed=best_config.seed,
        output_dir="./final_model",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        max_seq_length=best_config.max_seq_length,
        dataset_text_field="text",
        packing=True,
    ),
)

trainer.train()
print("\nFinal training complete!")

### Export to GGUF

In [ ]:
# Save LoRA adapter
model.save_pretrained("sentiment_lora")
tokenizer.save_pretrained("sentiment_lora")
print("LoRA adapter saved to sentiment_lora/")

# Export to GGUF (Q8_0 = good quality on CPU)
model.save_pretrained_gguf("sentiment_gguf", tokenizer, quantization_method="q8_0")
print("GGUF (Q8_0) saved to sentiment_gguf/")

# Also export q4_k_m for smaller file size
model.save_pretrained_gguf("sentiment_gguf_q4", tokenizer, quantization_method="q4_k_m")
print("GGUF (q4_k_m) saved to sentiment_gguf_q4/")

### Download Your Model

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("sentiment_gguf*/*.gguf")
print("Available GGUF files:")
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {f} ({size_mb:.0f} MB)")

q8_files = glob.glob("sentiment_gguf/*.gguf")
if q8_files:
    print(f"\nDownloading {q8_files[0]}...")
    files.download(q8_files[0])

## 8. Quick Test: Classify New Text

In [ ]:
from sentiment.data import format_for_inference

FastLanguageModel.for_inference(model)

test_texts = [
    "This coffee has a wonderful bitter edge with notes of dark chocolate",
    "The chocolate bar was way too bitter, almost inedible",
    "Smooth and creamy texture, exactly what I was looking for",
    "It tastes bland and watery, very disappointing",
]

test_category = None  # Set to e.g. "coffee" for context-aware classification

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

for text in test_texts:
    messages = format_for_inference(text, LABELS, category=test_category)
    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False,
    )
    inputs = tokenizer(
        input_text, return_tensors="pt", add_special_tokens=False,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=16, do_sample=False, use_cache=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(f"\n  Text: {text}")
    print(f"  Prediction: {prediction}")

## 9. Save Best Configuration

In [ ]:
best_config_dict = best_config.to_dict()
best_config_dict["best_macro_f1"] = best_metric
best_config_dict["labels"] = LABELS
best_config_dict["n_experiments"] = experiment_num - 1

with open("best_config.json", "w") as f:
    json.dump(best_config_dict, f, indent=2)

print("Best config saved to best_config.json:")
print(json.dumps(best_config_dict, indent=2))

files.download(RESULTS_FILE)
files.download("best_config.json")